<a href="https://colab.research.google.com/github/muajnstu/Customer_Churn_Prediction/blob/main/Employee_Attrition_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, NuSVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, roc_auc_score
from imblearn.over_sampling import SMOTE

In [2]:
def train_and_evaluate_models(models, X_train, X_test, y_train, y_test):
    results = {}
    for name, model in models.items():
        print(f"Training {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred)

        results[name] = {
            'accuracy': accuracy,
            'precision': precision,
            'f1_score': f1,
            'recall': recall,
            'auc': auc
        }

        print(f"{name} metrics:")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  AUC: {auc:.4f}")

    return results

# Data Loading

In [3]:
df=pd.read_csv('https://raw.githubusercontent.com/muajnstu/ML-Datasets/refs/heads/main/WA_Fn-UseC_-HR-Employee-Attrition.csv')

In [4]:
df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,No,Travel_Frequently,884,Research & Development,23,2,Medical,1,2061,...,3,80,1,17,3,3,5,2,0,3
1466,39,No,Travel_Rarely,613,Research & Development,6,1,Medical,1,2062,...,1,80,1,9,5,3,7,7,1,7
1467,27,No,Travel_Rarely,155,Research & Development,4,3,Life Sciences,1,2064,...,2,80,1,6,0,3,6,2,0,3
1468,49,No,Travel_Frequently,1023,Sales,2,3,Medical,1,2065,...,4,80,0,17,3,2,9,6,0,8


In [5]:
df.duplicated().sum()
print(df.duplicated().sum())

0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [6]:
df['Attrition'].value_counts()

,count
Attrition,
No,1233
Yes,237


Observation: it is an imbalanced dataset

In [7]:
for col in df.columns:
    print(f"Unique values for column '{col}':")
    print(df[col].unique())
    print("\n")

Unique values for column 'Age':
[41 49 37 33 27 32 59 30 38 36 35 29 31 34 28 22 53 24 21 42 44 46 39 43
 50 26 48 55 45 56 23 51 40 54 58 20 25 19 57 52 47 18 60]


Unique values for column 'Attrition':
['Yes' 'No']


Unique values for column 'BusinessTravel':
['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']


Unique values for column 'DailyRate':
[1102  279 1373 1392  591 1005 1324 1358  216 1299  809  153  670 1346
  103 1389  334 1123 1219  371  673 1218  419  391  699 1282 1125  691
  477  705  924 1459  125  895  813 1273  869  890  852 1141  464 1240
 1357  994  721 1360 1065  408 1211 1229  626 1434 1488 1097 1443  515
  853 1142  655 1115  427  653  989 1435 1223  836 1195 1339  664  318
 1225 1328 1082  548  132  746  776  193  397  945 1214  111  573 1153
 1400  541  432  288  669  530  632 1334  638 1093 1217 1353  120  682
  489  807  827  871  665 1040 1420  240 1280  534 1456  658  142 1127
 1031 1189 1354 1467  922  394 1312  750  441  684  249  841  147  528
  594  4

# Data Cleaning

1. Encoding
2. Dropping unnecessary colums

In [8]:
# Binary categorical mappings
attrition_mapping = {'No': 0, 'Yes': 1}
gender_mapping = {'Male': 0, 'Female': 1}
overtime_mapping = {'No': 0, 'Yes': 1}

# Nominal categorical mappings
business_travel_mapping = {
    'Non-Travel': 0,
    'Travel_Rarely': 1,
    'Travel_Frequently': 2
}

department_mapping = {
    'Sales': 0,
    'Research & Development': 1,
    'Human Resources': 2
}

education_field_mapping = {
    'Life Sciences': 0,
    'Medical': 1,
    'Marketing': 2,
    'Technical Degree': 3,
    'Human Resources': 4,
    'Other': 5
}

job_role_mapping = {
    'Sales Executive': 0,
    'Research Scientist': 1,
    'Laboratory Technician': 2,
    'Manufacturing Director': 3,
    'Healthcare Representative': 4,
    'Manager': 5,
    'Sales Representative': 6,
    'Research Director': 7,
    'Human Resources': 8
}

marital_status_mapping = {
    'Single': 0,
    'Married': 1,
    'Divorced': 2
}

# Apply mappings
df['Attrition'] = df['Attrition'].map(attrition_mapping)
df['Gender'] = df['Gender'].map(gender_mapping)
df['OverTime'] = df['OverTime'].map(overtime_mapping)

df['BusinessTravel'] = df['BusinessTravel'].map(business_travel_mapping)
df['Department'] = df['Department'].map(department_mapping)
df['EducationField'] = df['EducationField'].map(education_field_mapping)
df['JobRole'] = df['JobRole'].map(job_role_mapping)
df['MaritalStatus'] = df['MaritalStatus'].map(marital_status_mapping)

encoded_cols = [
    'Attrition',
    'Gender',
    'OverTime',
    'BusinessTravel',
    'Department',
    'EducationField',
    'JobRole',
    'MaritalStatus'
]

for col in encoded_cols:
    print(f"Unique values for column '{col}':")
    print(df[col].unique())
    print("\n")

Unique values for column 'Attrition':
[1 0]


Unique values for column 'Gender':
[1 0]


Unique values for column 'OverTime':
[1 0]


Unique values for column 'BusinessTravel':
[1 2 0]


Unique values for column 'Department':
[0 1 2]


Unique values for column 'EducationField':
[0 5 1 2 3 4]


Unique values for column 'JobRole':
[0 1 2 3 4 5 6 7 8]


Unique values for column 'MaritalStatus':
[0 1 2]




In [9]:
# Drop unnecessary columns
df.drop(['EmployeeCount',
         'Over18',
         'StandardHours',
         'EmployeeNumber'], axis=1, inplace=True)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   int64
 2   BusinessTravel            1470 non-null   int64
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   int64
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   int64
 8   EnvironmentSatisfaction   1470 non-null   int64
 9   Gender                    1470 non-null   int64
 10  HourlyRate                1470 non-null   int64
 11  JobInvolvement            1470 non-null   int64
 12  JobLevel                  1470 non-null   int64
 13  JobRole                   1470 non-null   int64
 14  JobSatisfaction           1470 non-null 

# Data Balancing

In [12]:
X = df.drop('Attrition', axis=1)
y = df['Attrition']

In [13]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
print("Class distribution after SMOTE:", pd.Series(y_resampled).value_counts())

Class distribution after SMOTE: Attrition
1    1233
0    1233
Name: count, dtype: int64


# Train-Test split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=46, stratify=y_resampled
)

In [17]:
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (1972, 30)
Shape of X_test: (494, 30)
Shape of y_train: (1972,)
Shape of y_test: (494,)


# Model Building

In [15]:
models = {
    "LogisticRegression": LogisticRegression(),
    "SVC": SVC(),
    "LinearSVC": LinearSVC(),
    "KNeighborsClassifier": KNeighborsClassifier(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "ExtraTreeClassifier": ExtraTreeClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
    "GradientBoostingClassifier": GradientBoostingClassifier(),
    "AdaBoostClassifier": AdaBoostClassifier(),
    "ExtraTreesClassifier": ExtraTreesClassifier(),
    "GaussianNB": GaussianNB(),
    "BernoulliNB": BernoulliNB(),
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "QuadraticDiscriminantAnalysis": QuadraticDiscriminantAnalysis()
}

# Evaluation

In [16]:
model_results = train_and_evaluate_models(models, X_train, X_test, y_train, y_test)


Training LogisticRegression...
LogisticRegression metrics:
  Accuracy: 0.7024
  Precision: 0.7033
  F1 Score: 0.7018
  Recall: 0.7004
  AUC: 0.7024
Training SVC...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


SVC metrics:
  Accuracy: 0.6053
  Precision: 0.5839
  F1 Score: 0.6499
  Recall: 0.7328
  AUC: 0.6053
Training LinearSVC...
LinearSVC metrics:
  Accuracy: 0.8320
  Precision: 0.8306
  F1 Score: 0.8323
  Recall: 0.8340
  AUC: 0.8320
Training KNeighborsClassifier...
KNeighborsClassifier metrics:
  Accuracy: 0.7591
  Precision: 0.7065
  F1 Score: 0.7864
  Recall: 0.8866
  AUC: 0.7591
Training DecisionTreeClassifier...
DecisionTreeClassifier metrics:
  Accuracy: 0.8057
  Precision: 0.7807
  F1 Score: 0.8140
  Recall: 0.8502
  AUC: 0.8057
Training ExtraTreeClassifier...
ExtraTreeClassifier metrics:
  Accuracy: 0.8279
  Precision: 0.8022
  F1 Score: 0.8350
  Recall: 0.8704
  AUC: 0.8279
Training RandomForestClassifier...
RandomForestClassifier metrics:
  Accuracy: 0.9271
  Precision: 0.9271
  F1 Score: 0.9271
  Recall: 0.9271
  AUC: 0.9271
Training GradientBoostingClassifier...
GradientBoostingClassifier metrics:
  Accuracy: 0.9109
  Precision: 0.9076
  F1 Score: 0.9113
  Recall: 0.9150
  AU